# Study 806 — Prospect-Theory Value — the teardown

The TK value math, the per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'sort_start': '2015-01', 'sort_end': '2026-05', 'n_months': 137, 'spread_bps': 138.7, 't_nw': 3.15, 't_1s': 2.87, 'lo_bps': 232.81, 'hi_bps': 94.11, 'welch_t': 2.05, 'gross_sharpe': 0.85, 'placebo_obs': 138.7, 'placebo_mean': 0.166, 'placebo_sd': 33.869, 'placebo_p': 0.0, 'placebo_sigma': 4.09, 'placebo_draws': 1000, 'era_early_bps': 100.35, 'era_early_t': 2.0, 'era_early_n': 60, 'era_late_bps': 168.58, 'era_late_t': 2.48, 'era_late_n': 77, 'timer_1_gross': 138.7, 'timer_1_cost': 6.17, 'timer_1_net': 132.53, 'timer_1_t': 2.74, 'timer_1_sharpe': 0.81, 'timer_1_ann': 15.9, 'timer_5_gross': 138.7, 'timer_5_cost': 14.17, 'timer_5_net': 124.53, 'timer_5_t': 2.57, 'timer_5_sharpe': 0.76, 'timer_5_ann': 14.9, 'null_mean_t': -0.11, 'null_sd_t': 1.01, 'null_fire': 0, 'planted_t': 2.82, 'planted_welch': 2.4}

## The TK value function — a quick live sanity check

A right-skewed, lottery-like distribution must score a **higher** TK value than its left-skewed mirror; the decision weights are non-negative and subadditive.

In [2]:
import os, sys, numpy as np
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from prospect_theory import strategy as st
rng = np.random.default_rng(2)
lottery = np.concatenate([rng.normal(-0.004,0.008,480), rng.normal(0.12,0.02,20)])
crash = -lottery
print(f'TK(lottery, right-skew) = {st.tk_value(lottery):+.4f}  (HIGH -> over-priced)')
print(f'TK(crash,   left-skew)  = {st.tk_value(crash):+.4f}  (LOW)')
assert st.tk_value(lottery) > st.tk_value(crash)

TK(lottery, right-skew) = -0.0021  (HIGH -> over-priced)
TK(crash,   left-skew)  = -0.0301  (LOW)


## The headline — long-low-TK / short-high-TK spread

Monthly equal-weight bottom-30% (low TK) minus top-30% (high TK) spread, 137 months.

In [3]:
print(f"spread        : {R['spread_bps']:+.2f} bps/month  NW(6) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : low-TK {R['lo_bps']:+.2f} vs high-TK {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (ann., before cost)")

spread        : +138.70 bps/month  NW(6) t = +3.15  one-sample t = +2.87
books         : low-TK +232.81 vs high-TK +94.11 bps (Welch t = +2.05)
gross Sharpe  : 0.85 (ann., before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [4]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.5f} "
      f"({R['placebo_sigma']:+.2f} sigma into the right tail)")

observed +138.70 bps vs placebo mean +0.166 (sd 33.869) -> p = 0.00000 (+4.09 sigma into the right tail)


## Robustness — two eras (split 2020-01-01)

In [5]:
print(f"2015-2019 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2020-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2015-2019 (n=60): +100.35 bps  NW t = +2.00
2020-2026 (n=77): +168.58 bps  NW t = +2.48


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per monthly rebalance; short pays 50 bps/yr borrow.

In [6]:
for tag,g,c,n,t,sh,an in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t'],R['timer_1_sharpe'],R['timer_1_ann']),
                          ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'],R['timer_5_sharpe'],R['timer_5_ann'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/mo (cost {c:.2f}/reb, t={t:+.2f}, Sharpe {sh:.2f}, ~{an:+.1f}%/yr)")

 1 bp one-way: gross +138.70 -> net +132.53 bps/mo (cost 6.17/reb, t=+2.74, Sharpe 0.81, ~+15.9%/yr)
5 bps one-way: gross +138.70 -> net +124.53 bps/mo (cost 14.17/reb, t=+2.57, Sharpe 0.76, ~+14.9%/yr)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation.

In [7]:
import numpy as np
from prospect_theory import data
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=806+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0020, seed=806, n_assets=40, n_days=1500))
print(f"planted (edge=0.0020): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean +0.15 (sd 1.03), |t|>=2 in 0/8


planted (edge=0.0020): NW t = +2.82, Welch t = +2.40


## Verdict

- **Signal — Real.** The Barberis-Mukherjee-Wang prospect-theory-value premium **replicates** on 50 liquid US mega-caps with the **predicted sign**: the long-low-TK / short-high-TK spread is **+138.70 bps/month** (NW *t* = **+3.15**), holds in both eras (*t* = +2.00 / +2.48), sits ~+4.1σ into the right tail of a 1,000-permutation placebo, and the 20-seed synthetic control recovers a *planted* relation cleanly (*t* = +2.82, fires on 0/20 nulls). Survivorship biases the magnitude (upper bound).
- **Tradability — Fragile.** The net edge *survives* conservative costs (net **+124.53 bps/month** at 5 bps one-way, *t* = +2.57, ~+14.9%/yr) — not a Mirage — but the magnitude is a survivorship upper bound (the short leg's blown-up lottery names are absent) and the 50-name universe concentrates the short into a few hard-to-borrow lottery mega-caps whose realistic borrow/squeeze exceeds the 50 bps/yr charged. Real signal, fragile paycheck.